# FinDER × RDF vs LPG: five evaluation tracks (fully embedded)

We build the same FinDER knowledge graph twice — once as **LPG** (labeled property graph backed by LanceDB) and once as **RDF / OWL** (backed by [owlready2](https://owlready2.readthedocs.io/), a Python OWL store that loads FIBO ontologies natively). Both run inside this Jupyter container — no graph server required.

**Why owlready2 for the RDF side?**

- FIBO is published as OWL ontologies; owlready2 loads them directly
- It exposes OWL classes and individuals as Python classes (so a node label maps to a class, an edge type maps to an `ObjectProperty`)
- It supports SPARQL via projection to rdflib for the comparison queries
- It can run an OWL reasoner (`sync_reasoner`) — the kind of inference LPG simply cannot perform
- This matches how the rest of seocho already uses owlready2 (offline ontology governance only — see `seocho/ontology_governance.py`)

**Five evaluation tracks:**

| Track | What it checks |
|---|---|
| 1. Golden Standard | Class/relationship overlap against the reference FIBO module set |
| 2. Data-Driven | Fraction of corpus-extracted entities the ontology has a class for |
| 3. Application/Task | FinDER QA correctness via Cypher-style retrieval (LPG) and SPARQL (RDF) |
| 4. User-based | Likert form — qualitative; emitted as a CSV scaffold |
| 5. Structure-based | Network metrics (LPG-only; RDF skipped because triples aren't first-class edges) |

**Bonus track: OWL reasoning.** After the five evaluations we run `sync_reasoner` over the OWL store and compare the inferred-triple count to the asserted count. This is the LPG-can't-do-this slot.

**Requirements.** All embedded — `owlready2`, `rdflib`, `lancedb`, `networkx`. No external services. The OWL reasoner step needs Java (HermiT); the notebook reports gracefully if no JVM is found.

## Setup

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")
LPG_WORKSPACE  = "finder_lpg"

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "finder" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_rdf_vs_lpg"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"FinDER:        {FINDER_PATH}")
print(f"Workdir:       {WORKDIR}")
print(f"LLM:           {LLM_PROVIDER}/{LLM_MODEL}")
print(f"LPG backend:   Neo4j @ {NEO4J_URI}")
print(f"RDF backend:   owlready2 (embedded)")

In [ ]:
from seocho.benchmarking import load_finder_cases
from seocho.index.pipeline import IndexingPipeline
from seocho.store.graph import Neo4jGraphStore
from seocho.store.llm import create_llm_backend

from examples.finder.datasets.fibo_modules.compose import compose_modules
from examples.finder.lib.owlready_graph_store import OwlreadyGraphStore
from examples.finder.lib.lpg_metrics import compute_lpg_structure_metrics
from examples.finder.lib.rdf_lpg_comparison import (
    corpus_coverage,
    golden_standard_overlap,
    task_track_aggregate,
    write_user_eval_template,
)

cases = load_finder_cases(FINDER_PATH)
llm = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)
print(f"Loaded {len(cases)} FinDER cases")

## Build LPG and RDF/OWL ontologies

Both sides use the **full** FIBO module composition (BE + FBC + SEC + FND + IND). The only difference is `graph_model` and the FIBO namespace passed to owlready2.

In [ ]:
lpg_ontology = compose_modules(["be", "fbc", "sec", "fnd", "ind"])
lpg_ontology.graph_model = "lpg"

rdf_ontology = compose_modules(["be", "fbc", "sec", "fnd", "ind"])
rdf_ontology.graph_model = "rdf"
rdf_ontology.namespace = "https://spec.edmcouncil.org/fibo/"

lpg_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
# Drop any leftover nodes from a previous run.
lpg_store.execute_write(
    "MATCH (n) WHERE n._workspace_id = $workspace_id DETACH DELETE n",
    params={"workspace_id": LPG_WORKSPACE},
)
lpg_store.ensure_constraints(lpg_ontology)

rdf_store = OwlreadyGraphStore(
    ontology=rdf_ontology,
    namespace=rdf_ontology.namespace,
    store_path=str(WORKDIR / "rdf.sqlite3"),
)

print(f"LPG schema:  {lpg_store.get_schema()}")
print(f"RDF schema:  {rdf_store.get_schema()}")

## Index FinDER documents into both stores

In [ ]:
captured = {"lpg": {"nodes": [], "rels": []}, "rdf": {"nodes": [], "rels": []}}

def make_pipeline(ontology, store, workspace_id, bucket):
    def cap(nodes, rels):
        bucket["nodes"].extend(nodes)
        bucket["rels"].extend(rels)
        return nodes, rels
    return IndexingPipeline(
        ontology=ontology,
        graph_store=store,
        llm=llm,
        workspace_id=workspace_id,
        on_after_extract=cap,
    )

for name, ontology, store, ws in (
    ("lpg", lpg_ontology, lpg_store, LPG_WORKSPACE),
    ("rdf", rdf_ontology, rdf_store, "finder_rdf"),
):
    print(f"== Building {name} graph ==")
    pipeline = make_pipeline(ontology, store, ws, captured[name])
    for case in cases:
        try:
            pipeline.index(case.text, metadata={"case_id": case.case_id})
        except Exception as exc:
            print(f"  index failure on {case.case_id}: {exc}")
    print(f"  captured {len(captured[name]['nodes'])} nodes, {len(captured[name]['rels'])} rels")

## Track 1 — Golden Standard overlap

In [ ]:
fibo_reference_classes = list(lpg_ontology.nodes.keys())

def constructed_classes(bucket):
    return {n.get("label", "") for n in bucket["nodes"] if n.get("label")}

track1 = {
    "lpg": golden_standard_overlap(constructed_classes(captured["lpg"]), fibo_reference_classes),
    "rdf": golden_standard_overlap(constructed_classes(captured["rdf"]), fibo_reference_classes),
}
for path, m in track1.items():
    print(f"  {path}: jaccard={m['jaccard']:.2f}  precision={m['precision']:.2f}  recall={m['recall']:.2f}")

## Track 2 — Data-Driven coverage

In [ ]:
alias_map = {
    label: list(node_def.aliases or [])
    for label, node_def in lpg_ontology.nodes.items()
}
extracted_class_names = lambda bucket: [str(n.get("label", "")) for n in bucket["nodes"] if n.get("label")]
track2 = {
    "lpg": corpus_coverage(extracted_class_names(captured["lpg"]), alias_map),
    "rdf": corpus_coverage(extracted_class_names(captured["rdf"]), alias_map),
}
for path, m in track2.items():
    print(f"  {path}: classified={m['classified_count']}/{m['total_distinct_entities']} coverage={m['coverage']:.2f}")

## Track 3 — Application/Task (FinDER QA)

We answer the FinDER questions through each store with a small retriever appropriate for that representation:
- **LPG:** keyword-anchored neighbor expansion on `LanceGraphStore` (same recipe as Tutorial 1).
- **RDF:** SPARQL `SELECT` against the OWL store (via `OwlreadyGraphStore.sparql`).

Both retrieved contexts feed the same OpenAI synthesizer so the comparison is apples-to-apples on the answer side.

In [ ]:
import re

STOPWORDS = {"what", "where", "who", "when", "how", "the", "a", "an", "is", "was", "were", "of", "in", "to", "for", "on", "and", "did", "during"}
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_.&\-]+")
ANSWER_SYSTEM = "Answer using only the supplied graph evidence. Be concise. If the evidence does not contain the answer, say so."

def question_keywords(question: str) -> list[str]:
    return [t for t in TOKEN_RE.findall(question) if t.lower() not in STOPWORDS]

def lpg_context(question: str) -> str:
    """Cypher-driven keyword + 1-hop expansion against the Neo4j LPG."""
    facts: list[str] = []
    seen: set[str] = set()
    for kw in question_keywords(question):
        seeds = lpg_store.query(
            "MATCH (n) WHERE n._workspace_id = $ws "
            "AND toLower(n.name) CONTAINS toLower($kw) "
            "RETURN n.id AS id, labels(n)[0] AS label, n.name AS name, properties(n) AS props "
            "LIMIT 3",
            params={"ws": LPG_WORKSPACE, "kw": kw},
        )
        for seed in seeds:
            if seed["id"] in seen:
                continue
            seen.add(seed["id"])
            facts.append(f"{seed['label']}({seed['name']}) properties={seed['props']}")
            hops = lpg_store.query(
                "MATCH (n {id: $id, _workspace_id: $ws})-[r]-(m) "
                "RETURN m.id AS neighbor_id, labels(m)[0] AS neighbor_label, "
                "       m.name AS neighbor_name, type(r) AS edge_type, "
                "       CASE WHEN startNode(r) = n THEN 'out' ELSE 'in' END AS direction "
                "LIMIT 5",
                params={"id": seed["id"], "ws": LPG_WORKSPACE},
            )
            for hop in hops:
                arrow = "->" if hop["direction"] == "out" else "<-"
                facts.append(
                    f"{seed['name']} {arrow}[{hop['edge_type']}]{arrow} "
                    f"{hop['neighbor_label']}({hop['neighbor_name']})"
                )
    return "\n".join(facts)

def rdf_context(question: str) -> str:
    """SPARQL-driven retrieval against the OWL/RDF store."""
    rows: list[str] = []
    for kw in question_keywords(question):
        sparql_query = f'''
PREFIX fibo: <{rdf_ontology.namespace}>
SELECT ?ind ?cls ?annot
WHERE {{
  ?ind a ?cls .
  OPTIONAL {{ ?ind fibo:has_property ?annot . }}
  FILTER(
    CONTAINS(LCASE(STR(?ind)), LCASE("{kw.lower()}")) ||
    CONTAINS(LCASE(COALESCE(STR(?annot), "")), LCASE("{kw.lower()}"))
  )
}}
LIMIT 5
'''
        try:
            for row in rdf_store.sparql(sparql_query):
                rows.append(f"{row.get('cls','')}: {row.get('ind','')} | {row.get('annot','')}")
        except Exception:
            continue
    return "\n".join(rows)

def answer(context: str, question: str) -> dict:
    if not context:
        return {"answer": "(no graph evidence)", "executed_ok": False}
    user = f"Graph evidence:\n{context}\n\nQuestion: {question}"
    response = llm.complete(system=ANSWER_SYSTEM, user=user)
    return {"answer": response.text.strip(), "executed_ok": True}

lpg_answers, rdf_answers = [], []
for case in cases:
    lpg_a = answer(lpg_context(case.question), case.question)
    rdf_a = answer(rdf_context(case.question), case.question)
    lpg_answers.append({"case_id": case.case_id, "question": case.question, "answer": lpg_a["answer"], "expected": case.expected_answer, "executed_ok": lpg_a["executed_ok"]})
    rdf_answers.append({"case_id": case.case_id, "question": case.question, "answer": rdf_a["answer"], "expected": case.expected_answer, "executed_ok": rdf_a["executed_ok"]})

track3 = {"lpg": task_track_aggregate(lpg_answers), "rdf": task_track_aggregate(rdf_answers)}
for path, m in track3.items():
    print(f"  {path}: contains_match={m['contains_match_rate']:.2f}  exec_success={m['exec_success_rate']:.2f}")

## Track 4 — User-based (CSV scaffold)

In [ ]:
user_eval_questions = [
    {
        "question": lpg_a["question"],
        "lpg_answer": lpg_a["answer"],
        "rdf_answer": rdf_a["answer"],
    }
    for lpg_a, rdf_a in zip(lpg_answers, rdf_answers)
]
scaffold_path = WORKDIR / "user_eval_template.csv"
write_user_eval_template(scaffold_path, questions=user_eval_questions, reviewers=5)
print(f"Wrote {scaffold_path}")
print("Distribute to reviewers, fill the Likert columns, then aggregate by mean score per path.")

## Track 5 — Structure-based (LPG only)

Network metrics on the LanceDB-backed property graph. RDF is skipped because owlready2 stores triples — without a property-graph projection, degree distribution and clustering aren't first-class on the RDF side.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx

# Pull the LPG workspace back from Neo4j into a NetworkX DiGraph so we
# can compute the structural metrics that LPG makes free.
lpg_metrics = compute_lpg_structure_metrics(lpg_store, database="neo4j", workspace_id=LPG_WORKSPACE)

deg_hist = lpg_metrics.get("degree_histogram", {})
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
if deg_hist:
    ax.bar(list(deg_hist.keys()), list(deg_hist.values()))
ax.set_title("LPG degree distribution")
ax.set_xlabel("degree")
ax.set_ylabel("node count")
plt.tight_layout()
plt.show()

track5 = {
    "lpg": {k: v for k, v in lpg_metrics.items() if k != "in_degree_histogram" and k != "out_degree_histogram"},
    "rdf": {"note": "skipped — RDF needs property-graph projection for native network metrics."},
}
print(json.dumps(track5, indent=2, default=str))

## Bonus — OWL reasoning (LPG can't do this)

Run the HermiT reasoner over the OWL store and compare asserted vs inferred triple counts. Skipped gracefully if no JVM is present (HermiT is Java).

In [ ]:
rdf_graph = rdf_store._world.as_rdflib_graph()
asserted = len(rdf_graph)
result = rdf_store.run_reasoner()
rdf_graph_after = rdf_store._world.as_rdflib_graph()
inferred = len(rdf_graph_after) - asserted
print(f"reasoner ok: {result.get('ok')} (skipped reason: {result.get('error', '-')})")
print(f"asserted triples: {asserted}")
print(f"inferred triples: {inferred} (added by sync_reasoner)")
print("\nA few inferred type assertions:")
for s, p, o in list(rdf_graph_after.triples((None, None, None)))[:6]:
    print(f"  {s} {p} {o}")

## Scoreboard

In [ ]:
scoreboard = {
    "track1_golden_standard": track1,
    "track2_data_driven": {
        "lpg": {"coverage": track2["lpg"]["coverage"]},
        "rdf": {"coverage": track2["rdf"]["coverage"]},
    },
    "track3_application_task": track3,
    "track4_user_based": {"note": f"populate {scaffold_path.name} and aggregate."},
    "track5_structure_based": track5,
    "bonus_owl_reasoning": {
        "asserted_triples": asserted,
        "inferred_triples": inferred,
        "reasoner_ok": result.get('ok', False),
    },
}
(WORKDIR / "scoreboard.json").write_text(json.dumps(scoreboard, indent=2))
print(json.dumps(scoreboard, indent=2))

## When to choose which

- **LPG (LanceDB)** — pick when you want network analytics (centrality, communities, paths), schema-light experimentation, or Cypher ergonomics. Track 5 is essentially free and the bonus reasoning track is irrelevant.
- **RDF / OWL (owlready2)** — pick when portability matters (FIBO, OWL, SHACL natively), when answers must cite IRIs to ground in a public ontology, or when downstream consumers expect SPARQL. Track 1 should consistently win on RDF if the ontology is faithfully imported, and the bonus reasoning track is RDF-only.

Both stores in this tutorial are embedded — no servers, no network ports. To scale up, swap `LanceGraphStore` for `Neo4jGraphStore` and `OwlreadyGraphStore` for the same `Neo4jGraphStore` configured with `Ontology.graph_model="rdf"` (which writes via neosemantics). The seocho APIs upstream of the stores stay identical.

Cleanup — both stores live under `.seocho/finder_rdf_vs_lpg/`:
```
rm -rf .seocho/finder_rdf_vs_lpg
```

## See both graphs side by side

The two backends store the same FinDER content in fundamentally different shapes. Drawing each makes the asymmetry concrete: the LPG (left) groups nodes by label and connects them with typed edges; the RDF view (right) is triple-flat — every fact is a (subject, predicate, object) edge.

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg, draw_rdf, fetch_lpg_subgraph
import matplotlib.pyplot as plt

lpg_nodes, lpg_rels = fetch_lpg_subgraph(lpg_store, workspace_id=LPG_WORKSPACE, limit=80)
fig_lpg = draw_lpg(lpg_nodes, lpg_rels, title="LPG (Neo4j) — labels color nodes; edges are typed", figsize=(11, 7))
plt.show()

fig_rdf = draw_rdf(rdf_store._world.as_rdflib_graph(), title="RDF (owlready2) — triples are flat", max_triples=200, figsize=(11, 7))
plt.show()

print(f"LPG:  {len(lpg_nodes)} nodes, {len(lpg_rels)} typed relationships")
print(f"RDF:  {len(rdf_store._world.as_rdflib_graph())} triples (asserted)")
print(f"\nNeo4j Browser:  http://localhost:7474   (login neo4j / tutorialspw)")
print(f"  cypher:       MATCH (n)-[r]->(m) WHERE n._workspace_id = '{LPG_WORKSPACE}' RETURN n,r,m LIMIT 50")